
# Pipeline de Tratamento de Dados — Estrutura Medalhão
### Horas-Aula Diária (INEP), PNAD Contínua/IBGE e Painel Estratégico de Priorização (MEC)

Este notebook implementa a ingestão e o tratamento das bases externas em três camadas:

- **Bronze**: dados brutos, apenas com cabeçalho identificado e linhas de rodapé/fonte removidas. Nenhuma regra de negócio é aplicada aqui.
- **Silver**: dados limpos, tipados, com nomenclatura padronizada e **chaves geográficas completas** (`COD_UF`, `SG_UF`, `COD_REGIAO`) mesmo nas bases que originalmente não traziam essas colunas. Séries anuais são empilhadas.
- **Gold**: tabelas de fato, já cruzadas entre fontes e no grão (UF, Município, Escola) prontas para alimentar a modelagem supervisionada.

## Como usar
1. Ajuste a variável `INPUT_DIR` na célula de configuração abaixo para a pasta onde estão os arquivos originais.
2. Rode as células em ordem (Run All). As pastas `bronze/`, `silver/` e `gold/` são criadas automaticamente dentro de `OUTPUT_DIR`.
3. Dependências: `pandas`, `openpyxl`, `xlrd` (para os .xls antigos) e `pyarrow` (opcional, para salvar em parquet). Instale com:
   ```
   pip install pandas openpyxl xlrd pyarrow
   ```

## Arquivos de entrada esperados (nomes exatamente como fornecidos)
- `MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx`, `MÉDIA HAD_BRASIL_REGIOES_UFS_2024.xlsx`, `MÉDIA HAD_BRASIL_REGIOES_UFS_2025.xlsx`
- `MÉDIA HAD_MUNICIPIOS_2023.xlsx`, `MÉDIA HAD_MUNICIPIOS_2024.xlsx`, `MÉDIA HAD_MUNICIPIOS_2025.xlsx`
- `MÉDIA HAD_ESCOLAS_2023.xlsx`, `MÉDIA HAD_ESCOLAS_2024.xlsx`, `MÉDIA HAD_ESCOLAS_2025.xlsx`
- `Microdados do Painel Estratégico de Priorização.xlsx`
- `NOTA_TÉCNICA_PAINEL_PRIORIZAÇÃO.pdf` (usada apenas como referência para o de-para do risco, não é lida pelo notebook)
- `Tabela 2.2 (RendCaract_Geo_sbenef).xls`
- `Tabela 4.3 (FreqLiq_Geo).xls`
- `Tabela 4636 Rendimento médio mensal uf.xlsx`
- `Tabela 7109 População residente 6a9 anos.xlsx`
- `Tabela 7113  Taxa de analfabetismo.xlsx`



## 0. Checagem de dependências

Confere se `pandas`, `openpyxl` (leitura de `.xlsx`) e `xlrd` (leitura dos
`.xls` antigos — Tabelas 2.2 e 4.3) estão instalados **no mesmo interpretador
Python que este kernel está usando** (`sys.executable`), e instala
automaticamente o que faltar.

> Por que isso importa: `pip install` num terminal comum pode apontar para um
> Python diferente do kernel selecionado no notebook (é comum ter Anaconda,
> Python da Microsoft Store, venvs, etc. todos instalados). Rodar a instalação
> com `sys.executable -m pip install ...` garante que cai exatamente no
> ambiente que o notebook está de fato usando.


In [4]:

import sys
import subprocess
import importlib
from pathlib import Path

dependencias = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",   # necessário para ler .xlsx
    "xlrd": "xlrd>=2.0.1",    # necessário para ler .xls (Tabela 2.2 e 4.3)
    "pyarrow": "pyarrow",     # necessário para salvar/ler .parquet
}

def salvar_parquet(df, caminho):
    '''Salva um DataFrame em .parquet, garantindo nomes de coluna string
    (Parquet exige isso; leituras com header=None geram colunas inteiras
    0, 1, 2... que precisam virar "0", "1", "2"...) e tipagem homogênea por
    coluna (colunas "object" que misturam número e texto — ex.: os
    indicadores de HAD antes de tratar o marcador "--" — quebram a escrita
    em parquet, que exige um único tipo por coluna; aqui elas são
    convertidas para string, preservando nulos).'''
    df = df.rename(columns=str).copy()
    for col in df.columns:
        if df[col].dtype == "object":
            nulos = df[col].isna()
            df[col] = df[col].astype(str)
            df.loc[nulos, col] = None
    df.to_parquet(caminho, index=False)


print("Interpretador Python deste kernel:", sys.executable)

faltando = []
for modulo, pacote_pip in dependencias.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltando.append(pacote_pip)

if faltando:
    print(f"Pacote(s) ausente(s) neste kernel: {faltando}. Instalando automaticamente...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltando])
    importlib.invalidate_caches()

    ainda_faltando = []
    for pacote_pip in faltando:
        modulo = pacote_pip.split(">=")[0].split("==")[0]
        try:
            importlib.import_module(modulo)
        except ImportError:
            ainda_faltando.append(pacote_pip)

    if ainda_faltando:
        raise ModuleNotFoundError(
            f"Não consegui instalar automaticamente: {ainda_faltando}.\n"
            f"Rode manualmente no terminal, usando ESTE interpretador específico:\n\n"
            f'    "{sys.executable}" -m pip install {" ".join(ainda_faltando)}\n\n'
            "Depois reinicie o kernel (Restart) e rode as células de novo."
        )
    print("Instalado com sucesso. Pode continuar.")
else:
    print("OK — pandas, openpyxl e xlrd já estão instalados neste kernel.")


Interpretador Python deste kernel: c:\Users\jack_\OneDrive\Área de Trabalho\TECH CHALLENGE 3\fiap_tech_challenge_03\.venv\Scripts\python.exe
Pacote(s) ausente(s) neste kernel: ['openpyxl', 'xlrd>=2.0.1']. Instalando automaticamente...
Instalado com sucesso. Pode continuar.


In [5]:

import os
import re
import unicodedata
import warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
BASE = Path.cwd()

INPUT_DIR = BASE.parent / 'src' / 'data' / 'raw'



#INPUT_DIR = Path("C:/Users/jack_/OneDrive/Área de Trabalho/TECH CHALLENGE 3/dados_originais")

# Pasta raiz onde as camadas bronze/silver/gold serão criadas
#OUTPUT_DIR = Path("C:/Users/jack_/OneDrive/Área de Trabalho/TECH CHALLENGE 3/medalhao")

OUTPUT_DIR = INPUT_DIR.parent

BRONZE_DIR = OUTPUT_DIR / "bronze"
SILVER_DIR = OUTPUT_DIR / "silver"
SILVER_DIM_DIR = SILVER_DIR / "dim"
GOLD_DIR = OUTPUT_DIR / "gold"

for d in [BRONZE_DIR, SILVER_DIR, SILVER_DIM_DIR, GOLD_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def _lista_pasta(pasta):
    try:
        return os.listdir(pasta)
    except OSError:
        return []


def encontra_arquivo(nome_esperado):
    '''Resolve o caminho de um arquivo em INPUT_DIR de forma tolerante a:
    - normalização Unicode diferente para acentos (comum entre Mac/Windows/web,
      ex.: "É" pode ser salvo como 1 caractere composto ou como "E" + acento
      combinante — visualmente idênticos, mas bytes diferentes);
    - maiúsculas/minúsculas diferentes;
    - arquivos do OneDrive ainda não baixados localmente (placeholder "somente
      online"), que aparecem na listagem e passam no is_file(), mas falham ao
      serem efetivamente abertos.
    Sempre tenta de fato ABRIR o arquivo antes de devolver o caminho — por
    isso é seguro chamar em todo lugar que hoje usa pd.read_excel(inp(...)):
    se o arquivo não puder ser lido, o erro já sai com diagnóstico completo
    aqui, em vez do FileNotFoundError genérico do pandas lá na frente.
    '''
    candidato = INPUT_DIR / nome_esperado
    caminho_resolvido = None
    if candidato.is_file():
        caminho_resolvido = candidato
    else:
        candidatos = _lista_pasta(INPUT_DIR)
        alvo_norm = unicodedata.normalize("NFC", nome_esperado).lower().strip()
        alvo_norm = re.sub(r"\s+", " ", alvo_norm)
        for c in candidatos:
            candidato_norm = re.sub(r"\s+", " ", unicodedata.normalize("NFC", c).lower().strip())
            if candidato_norm == alvo_norm:
                caminho_resolvido = INPUT_DIR / c
                break

    if caminho_resolvido is not None:
        try:
            with open(caminho_resolvido, "rb") as fh:
                fh.read(4)
            return str(caminho_resolvido)
        except OSError as e:
            raise FileNotFoundError(
                f"O arquivo '{nome_esperado}' aparece na pasta "
                f"({caminho_resolvido}) mas não pôde ser aberto ({e}).\n\n"
                "Isso é típico de arquivos do OneDrive em modo 'somente online' "
                "(baixados sob demanda): o Windows mostra o arquivo como existente, "
                "mas o conteúdo ainda não está no disco.\n"
                "Solução: no Explorer do Windows, clique com o botão direito na "
                "pasta que contém os arquivos originais e escolha "
                "'Sempre manter neste dispositivo'. Espere o download terminar "
                "(ícone de nuvem vira um check verde) e rode o notebook de novo."
            ) from e

    listagem = "\n".join(f"  - {c!r}" for c in _lista_pasta(INPUT_DIR)) or "  (pasta vazia ou inexistente)"
    raise FileNotFoundError(
        f"Não encontrei o arquivo '{nome_esperado}' em {INPUT_DIR.resolve()}.\n\n"
        f"Arquivos encontrados nessa pasta:\n{listagem}\n\n"
        "Causas mais comuns:\n"
        "  1) Pasta dentro do OneDrive com o arquivo em modo 'somente online' "
        "(ícone de nuvem, não baixado de fato) — clique com o botão direito na "
        "pasta e escolha 'Sempre manter neste dispositivo'.\n"
        "  2) Nome do arquivo com acentuação salva de forma diferente.\n"
        "  3) Espaço extra no início/fim do nome do arquivo/pasta.\n"
        "  4) INPUT_DIR apontando para a pasta errada."
    )


def inp(filename):
    '''Monta (e valida) o caminho completo de um arquivo de entrada.'''
    return encontra_arquivo(filename)


print("Pasta de entrada (INPUT_DIR):", INPUT_DIR.resolve())
print("Diretórios de saída preparados em:", OUTPUT_DIR.resolve())


Pasta de entrada (INPUT_DIR): C:\Users\jack_\OneDrive\Área de Trabalho\TECH CHALLENGE 3\fiap_tech_challenge_03\src\data\raw
Diretórios de saída preparados em: C:\Users\jack_\OneDrive\Área de Trabalho\TECH CHALLENGE 3\fiap_tech_challenge_03\src\data



### Checagem rápida dos arquivos de entrada

Roda antes do resto do pipeline para falhar com uma mensagem clara em
português — incluindo a listagem real da pasta e as causas mais comuns
(placeholder do OneDrive, acentuação, etc.) — em vez de um erro genérico do
pandas lá na frente.


In [6]:
INPUT_DIR

WindowsPath('c:/Users/jack_/OneDrive/Área de Trabalho/TECH CHALLENGE 3/fiap_tech_challenge_03/src/data/raw')

In [7]:

arquivos_esperados = [
    "MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx",
    "MÉDIA HAD_BRASIL_REGIOES_UFS_2024.xlsx",
    "MÉDIA HAD_BRASIL_REGIOES_UFS_2025.xlsx",
    "MÉDIA HAD_MUNICIPIOS_2023.xlsx",
    "MÉDIA HAD_MUNICIPIOS_2024.xlsx",
    "MÉDIA HAD_MUNICIPIOS_2025.xlsx",
    "MÉDIA HAD_ESCOLAS_2023.xlsx",
    "MÉDIA HAD_ESCOLAS_2024.xlsx",
    "MÉDIA HAD_ESCOLAS_2025.xlsx",
    "Microdados do Painel Estratégico de Priorização.xlsx",
    "Tabela 2.2 (RendCaract_Geo_sbenef).xls",
    "Tabela 4.3 (FreqLiq_Geo).xls",
    "Tabela 4636 Rendimento médio mensal uf.xlsx",
    "Tabela 7109 População residente 6a9 anos.xlsx",
    "Tabela 7113  Taxa de analfabetismo.xlsx",
]

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"A pasta INPUT_DIR não existe: {INPUT_DIR.resolve()}\n"
        f"Crie a pasta e coloque os 15 arquivos originais nela, ou ajuste "
        f"a variável INPUT_DIR na célula anterior para o caminho correto."
    )

erros = []
for f in arquivos_esperados:
    try:
        encontra_arquivo(f)
    except Exception as e:
        erros.append(str(e))

if erros:
    raise FileNotFoundError("\n\n".join(erros))

print("OK — todos os", len(arquivos_esperados), "arquivos de entrada foram encontrados e puderam ser abertos.")


OK — todos os 15 arquivos de entrada foram encontrados e puderam ser abertos.



## 1. Camada Silver — Dimensões de apoio

Construímos três dimensões que não existem em nenhuma base isoladamente, mas
são necessárias para cruzar tudo:

- **`dim_uf`**: código IBGE de cada Unidade da Federação, sigla, nome, região e capital.
  Usada para preencher `COD_UF`/`SG_UF` nas bases que só trazem o **nome** da UF
  (ex.: `MEDIA_HAD_BRASIL_REGIOES_UFS_*`) ou o nome da **capital**
  (ex.: tabelas do IBGE/PNAD).
- **`dim_regiao`**: código e nome de cada grande região (ordem oficial IBGE).
- **`dim_risco`**: de-para da variável `Risco` (escala 1–5) usada no Painel Estratégico
  de Priorização, **conforme a Nota Técnica nº 3/2026/CGMAB/DIMAM/SEB/SEB, item 4.3.2.9**
  (classificação final de risco, produto de probabilidade × impacto).


In [8]:

# ----------------------------------------------------------------------
# dim_uf: código IBGE, sigla, nome, região e capital de cada UF
# ----------------------------------------------------------------------
dim_uf = pd.DataFrame([
    (11, "RO", "Rondônia",            "Norte",        "Porto Velho"),
    (12, "AC", "Acre",                "Norte",        "Rio Branco"),
    (13, "AM", "Amazonas",            "Norte",        "Manaus"),
    (14, "RR", "Roraima",             "Norte",        "Boa Vista"),
    (15, "PA", "Pará",                "Norte",        "Belém"),
    (16, "AP", "Amapá",               "Norte",        "Macapá"),
    (17, "TO", "Tocantins",           "Norte",        "Palmas"),
    (21, "MA", "Maranhão",            "Nordeste",     "São Luís"),
    (22, "PI", "Piauí",               "Nordeste",     "Teresina"),
    (23, "CE", "Ceará",               "Nordeste",     "Fortaleza"),
    (24, "RN", "Rio Grande do Norte", "Nordeste",     "Natal"),
    (25, "PB", "Paraíba",             "Nordeste",     "João Pessoa"),
    (26, "PE", "Pernambuco",          "Nordeste",     "Recife"),
    (27, "AL", "Alagoas",             "Nordeste",     "Maceió"),
    (28, "SE", "Sergipe",             "Nordeste",     "Aracaju"),
    (29, "BA", "Bahia",               "Nordeste",     "Salvador"),
    (31, "MG", "Minas Gerais",        "Sudeste",      "Belo Horizonte"),
    (32, "ES", "Espírito Santo",      "Sudeste",      "Vitória"),
    (33, "RJ", "Rio de Janeiro",      "Sudeste",      "Rio de Janeiro"),
    (35, "SP", "São Paulo",           "Sudeste",      "São Paulo"),
    (41, "PR", "Paraná",              "Sul",          "Curitiba"),
    (42, "SC", "Santa Catarina",      "Sul",          "Florianópolis"),
    (43, "RS", "Rio Grande do Sul",   "Sul",          "Porto Alegre"),
    (50, "MS", "Mato Grosso do Sul",  "Centro-Oeste", "Campo Grande"),
    (51, "MT", "Mato Grosso",         "Centro-Oeste", "Cuiabá"),
    (52, "GO", "Goiás",               "Centro-Oeste", "Goiânia"),
    (53, "DF", "Distrito Federal",    "Centro-Oeste", "Brasília"),
], columns=["COD_UF", "SG_UF", "NO_UF", "NO_REGIAO", "NO_CAPITAL"])

# ----------------------------------------------------------------------
# dim_regiao: código oficial IBGE (Norte=1 ... Centro-Oeste=5)
# ----------------------------------------------------------------------
ordem_regiao = {"Norte": 1, "Nordeste": 2, "Sudeste": 3, "Sul": 4, "Centro-Oeste": 5}
dim_regiao = (
    dim_uf[["NO_REGIAO"]]
    .drop_duplicates()
    .assign(COD_REGIAO=lambda d: d["NO_REGIAO"].map(ordem_regiao))
    [["COD_REGIAO", "NO_REGIAO"]]
    .sort_values("COD_REGIAO")
    .reset_index(drop=True)
)

# ----------------------------------------------------------------------
# dim_risco: de-para da variável Risco (1-5), Nota Técnica item 4.3.2.9
# ----------------------------------------------------------------------
dim_risco = pd.DataFrame([
    (1, "Muito baixo", "1 – Baixo risco de não alcançar a meta",
     "Poucos estudantes impactados"),
    (2, "Baixo", "2 – Risco moderado-baixo de não alcançar a meta",
     "Impacto em um grupo reduzido de estudantes"),
    (3, "Moderado", "3 – Risco moderado de não alcançar a meta",
     "Impacto em um grupo médio de estudantes"),
    (4, "Alto", "4 – Risco moderado-alto de não alcançar a meta",
     "Muitos estudantes impactados"),
    (5, "Muito alto", "5 – Alto risco de não alcançar a meta",
     "Grande número de estudantes impactados (mais de 80% em número de estudantes)"),
], columns=["COD_RISCO", "CATEGORIA_RISCO", "NIVEL_RISCO_DESC", "DESC_IMPACTO"])

salvar_parquet(dim_uf, os.path.join(SILVER_DIM_DIR, "dim_uf.parquet"))
salvar_parquet(dim_regiao, os.path.join(SILVER_DIM_DIR, "dim_regiao.parquet"))
salvar_parquet(dim_risco, os.path.join(SILVER_DIM_DIR, "dim_risco.parquet"))

# Estruturas de apoio reutilizadas nas próximas seções
UF_MAP = dict(zip(dim_uf["NO_UF"], dim_uf["COD_UF"]))
CAP_MAP = dict(zip(dim_uf["NO_CAPITAL"], dim_uf["COD_UF"]))
SG_TO_COD = dict(zip(dim_uf["SG_UF"], dim_uf["COD_UF"]))
COD_TO_SG = dict(zip(dim_uf["COD_UF"], dim_uf["SG_UF"]))
REGIOES = set(["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"])
GEO_VALIDOS = set(["Brasil"]) | REGIOES | set(UF_MAP.keys()) | set(CAP_MAP.keys())

dim_uf


,COD_UF,SG_UF,NO_UF,NO_REGIAO,NO_CAPITAL
0,11,RO,Rondônia,Norte,Porto Velho
1,12,AC,Acre,Norte,Rio Branco
2,13,AM,Amazonas,Norte,Manaus
3,14,RR,Roraima,Norte,Boa Vista
4,15,PA,Pará,Norte,Belém
5,16,AP,Amapá,Norte,Macapá
6,17,TO,Tocantins,Norte,Palmas
7,21,MA,Maranhão,Nordeste,São Luís
8,22,PI,Piauí,Nordeste,Teresina
9,23,CE,Ceará,Nordeste,Fortaleza



## 2. Função auxiliar — classificação e chave geográfica

Várias bases (Brasil/Regiões/UFs do INEP e as tabelas do IBGE/PNAD) misturam,
numa única coluna de texto, o nome do Brasil, das regiões, das UFs e, em
alguns casos, das capitais. A função abaixo:

1. Classifica cada linha em `BRASIL` / `REGIAO` / `UF` / `CAPITAL`.
2. Preenche `COD_UF` e `SG_UF` a partir do nome da UF **ou** do nome da capital,
   usando as dimensões construídas acima — é o "tratamento das que não possuem
   COD_UF" pedido.


In [9]:

def classifica_geo(nome):
    '''Classifica um rótulo textual de unidade geográfica.'''
    if pd.isna(nome):
        return None
    nome = str(nome).strip()
    if nome == "Brasil":
        return "BRASIL"
    if nome in REGIOES:
        return "REGIAO"
    if nome in UF_MAP:
        return "UF"
    if nome in CAP_MAP:
        return "CAPITAL"
    # algumas tabelas do IBGE trazem a capital como "Nome (SG)"
    if re.search(r"\([A-Z]{2}\)$", nome):
        return "CAPITAL_COM_SIGLA"
    return None


def adiciona_chaves_geo(df, col_geo="UNIDADE_GEO"):
    '''Adiciona TP_UNIDGEO, COD_UF e SG_UF a partir da coluna de texto col_geo.
    Cobre tanto capitais no formato 'Nome' quanto 'Nome (SG)'.'''
    df = df.copy()
    df["TP_UNIDGEO"] = df[col_geo].apply(classifica_geo)

    df["COD_UF"] = pd.NA
    df["SG_UF"] = pd.NA

    # UF: nome bate direto com dim_uf.NO_UF
    m_uf = df["TP_UNIDGEO"] == "UF"
    df.loc[m_uf, "COD_UF"] = df.loc[m_uf, col_geo].map(UF_MAP)

    # Capital "pura" (sem sigla): nome bate com dim_uf.NO_CAPITAL
    m_cap = df["TP_UNIDGEO"] == "CAPITAL"
    df.loc[m_cap, "COD_UF"] = df.loc[m_cap, col_geo].map(CAP_MAP)

    # Capital no formato "Nome (SG)": extrai a sigla entre parênteses
    m_cap_sg = df["TP_UNIDGEO"] == "CAPITAL_COM_SIGLA"
    sigla = df.loc[m_cap_sg, col_geo].astype(str).str.extract(r"\(([A-Z]{2})\)$")[0]
    df.loc[m_cap_sg, "SG_UF"] = sigla.values
    df.loc[m_cap_sg, "COD_UF"] = df.loc[m_cap_sg, "SG_UF"].map(SG_TO_COD)
    df.loc[m_cap_sg, "TP_UNIDGEO"] = "CAPITAL"

    df["COD_UF"] = pd.to_numeric(df["COD_UF"], errors="coerce")
    df.loc[df["SG_UF"].isna() & df["COD_UF"].notna(), "SG_UF"] = (
        df.loc[df["SG_UF"].isna() & df["COD_UF"].notna(), "COD_UF"].map(COD_TO_SG)
    )
    return df



## 3. HAD — Brasil, Regiões e UFs (2023-2025)

Empilha os três anos (`MEDIA_HAD_BRASIL_REGIOES_UFS_2023/2024/2025.xlsx`,
aba `BRASIL_REGIOES_UFS`), remove linhas de rodapé (Fonte/Nota), converte o
marcador `"--"` em nulo e converte os indicadores para numérico.

Esta base só traz o **nome** da unidade geográfica (`UNIDGEO`), então
`adiciona_chaves_geo` é usada para obter `COD_UF`/`SG_UF`/região.


In [10]:

arquivos_had_uf = {
    2023: "MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx",
    2024: "MÉDIA HAD_BRASIL_REGIOES_UFS_2024.xlsx",
    2025: "MÉDIA HAD_BRASIL_REGIOES_UFS_2025.xlsx",
}

frames = []
for ano, arq in arquivos_had_uf.items():
    df = pd.read_excel(inp(arq), sheet_name="BRASIL_REGIOES_UFS", header=8)
    df = df.dropna(how="all")
    df["_SRC_FILE"] = arq
    frames.append(df)

bronze_had_uf = pd.concat(frames, ignore_index=True)
# remove rodapé (linhas de Fonte/Nota que vêm na última posição do NU_ANO_CENSO)
bronze_had_uf = bronze_had_uf[
    ~bronze_had_uf["NU_ANO_CENSO"].astype(str).str.contains("Fonte|Nota", na=False)
]
bronze_had_uf["NU_ANO_CENSO"] = pd.to_numeric(bronze_had_uf["NU_ANO_CENSO"], errors="coerce")
bronze_had_uf = bronze_had_uf.dropna(subset=["NU_ANO_CENSO"])
salvar_parquet(bronze_had_uf, os.path.join(BRONZE_DIR, "had_brasil_regioes_ufs_2023_2025.parquet"))

# ---------------------------- SILVER ----------------------------------
silver_had_uf = bronze_had_uf.rename(columns={"NU_ANO_CENSO": "ANO"}).copy()
silver_had_uf = adiciona_chaves_geo(silver_had_uf, col_geo="UNIDGEO")

# para linhas de Região, preenche NO_REGIAO com o próprio nome
silver_had_uf["NO_REGIAO"] = pd.NA
silver_had_uf.loc[silver_had_uf["TP_UNIDGEO"] == "REGIAO", "NO_REGIAO"] = silver_had_uf["UNIDGEO"]
silver_had_uf = silver_had_uf.merge(
    dim_uf[["COD_UF", "NO_REGIAO", "NO_CAPITAL"]], on="COD_UF", how="left", suffixes=("", "_uf")
)
silver_had_uf["NO_REGIAO"] = silver_had_uf["NO_REGIAO"].fillna(silver_had_uf["NO_REGIAO_uf"])
silver_had_uf = silver_had_uf.drop(columns=["NO_REGIAO_uf"])

ind_cols = [c for c in silver_had_uf.columns if c.endswith("_CAT_0")]
for c in ind_cols:
    silver_had_uf[c] = pd.to_numeric(silver_had_uf[c].astype(str).replace("--", pd.NA), errors="coerce")

silver_had_uf = silver_had_uf.rename(
    columns={"NO_CATEGORIA": "LOCALIZACAO", "NO_DEPENDENCIA": "DEPENDENCIA_ADM"}
)

cols_final = (
    ["ANO", "TP_UNIDGEO", "UNIDGEO", "COD_REGIAO", "NO_REGIAO", "COD_UF", "SG_UF", "NO_CAPITAL",
     "LOCALIZACAO", "DEPENDENCIA_ADM"] + ind_cols + ["_SRC_FILE"]
)
silver_had_uf = silver_had_uf.merge(dim_regiao, on="NO_REGIAO", how="left")
silver_had_uf = silver_had_uf[[c for c in cols_final if c in silver_had_uf.columns]]

salvar_parquet(silver_had_uf, os.path.join(SILVER_DIR, "had_uf_regiao_brasil_2023_2025.parquet"))

print("HAD Brasil/Regiões/UFs:", silver_had_uf.shape)
print("Linhas de UF sem COD_UF:",
      silver_had_uf.loc[silver_had_uf["TP_UNIDGEO"] == "UF", "COD_UF"].isna().sum())
silver_had_uf.head()


HAD Brasil/Regiões/UFs: (1764, 32)
Linhas de UF sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDGEO,COD_REGIAO,NO_REGIAO,COD_UF,SG_UF,NO_CAPITAL,LOCALIZACAO,DEPENDENCIA_ADM,...,FUN_07_CAT_0,FUN_08_CAT_0,FUN_09_CAT_0,MED_CAT_0,MED_01_CAT_0,MED_02_CAT_0,MED_03_CAT_0,MED_04_CAT_0,MED_NS_CAT_0,_SRC_FILE
0,2023,BRASIL,Brasil,NaN,NaN,NaN,<NA>,NaN,Total,Total,...,5.3,5.2,5.4,5.7,5.9,5.7,5.5,5.0,4.5,MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx
1,2023,BRASIL,Brasil,NaN,NaN,NaN,<NA>,NaN,Urbana,Total,...,5.3,5.3,5.4,5.7,5.9,5.7,5.5,4.7,4.3,MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx
2,2023,BRASIL,Brasil,NaN,NaN,NaN,<NA>,NaN,Rural,Total,...,5.2,5.2,5.4,5.6,5.8,5.5,5.4,8.2,5.7,MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx
3,2023,BRASIL,Brasil,NaN,NaN,NaN,<NA>,NaN,Total,Federal,...,5.2,5.3,5.3,7.4,7.6,7.6,7.5,5.4,6.1,MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx
4,2023,BRASIL,Brasil,NaN,NaN,NaN,<NA>,NaN,Urbana,Federal,...,5.2,5.3,5.3,7.2,7.4,7.5,7.3,5.3,5.8,MEDIA HAD_BRASIL_REGIOES_UFS_2023.xlsx



## 4. HAD — Municípios (2023-2025)

Esta base já traz `SG_UF` e `CO_MUNICIPIO` (código IBGE de 7 dígitos), então
o enriquecimento aqui é apenas o `merge` com `dim_uf` para trazer `COD_UF`,
`NO_UF` e `COD_REGIAO`.


In [12]:

arquivos_had_mun = {
    2023: "MÉDIA HAD_MUNICIPIOS_2023.xlsx",
    2024: "MÉDIA HAD_MUNICIPIOS_2024.xlsx",
    2025: "MÉDIA HAD_MUNICIPIOS_2025.xlsx",
}

frames = []
for ano, arq in arquivos_had_mun.items():
    df = pd.read_excel(inp(arq), sheet_name="MUNICIPIO", header=8)
    df["_SRC_FILE"] = arq
    frames.append(df)

bronze_had_mun = pd.concat(frames, ignore_index=True)
bronze_had_mun = bronze_had_mun[
    ~bronze_had_mun["NU_ANO_CENSO"].astype(str).str.contains("Fonte|Nota", na=False)
]
bronze_had_mun["NU_ANO_CENSO"] = pd.to_numeric(bronze_had_mun["NU_ANO_CENSO"], errors="coerce")
bronze_had_mun = bronze_had_mun.dropna(subset=["NU_ANO_CENSO"])
salvar_parquet(bronze_had_mun, os.path.join(BRONZE_DIR, "had_municipios_2023_2025.parquet"))

# ---------------------------- SILVER ----------------------------------
silver_had_mun = bronze_had_mun.rename(columns={"NU_ANO_CENSO": "ANO"}).copy()
silver_had_mun = silver_had_mun.merge(dim_uf, on="SG_UF", how="left")

ind_cols_mun = [c for c in silver_had_mun.columns if "_CAT_0" in c]
for c in ind_cols_mun:
    silver_had_mun[c] = pd.to_numeric(silver_had_mun[c].astype(str).replace("--", pd.NA), errors="coerce")

silver_had_mun = silver_had_mun.rename(
    columns={"NO_REGIAO_x": "NO_REGIAO", "NO_CATEGORIA": "LOCALIZACAO", "NO_DEPENDENCIA": "DEPENDENCIA_ADM"}
)
cols_final = (
    ["ANO", "COD_REGIAO", "NO_REGIAO", "COD_UF", "SG_UF", "NO_UF", "CO_MUNICIPIO", "NO_MUNICIPIO",
     "LOCALIZACAO", "DEPENDENCIA_ADM"] + ind_cols_mun + ["_SRC_FILE"]
)
silver_had_mun = silver_had_mun[[c for c in cols_final if c in silver_had_mun.columns]]

salvar_parquet(silver_had_mun, os.path.join(SILVER_DIR, "had_municipios_2023_2025.parquet"))

print("HAD Municípios:", silver_had_mun.shape)
print("Linhas sem COD_UF:", silver_had_mun["COD_UF"].isna().sum())
silver_had_mun.head()


HAD Municípios: (198901, 31)
Linhas sem COD_UF: 0


,ANO,NO_REGIAO,COD_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,LOCALIZACAO,DEPENDENCIA_ADM,ED_INF_CAT_0,...,FUN_07_CAT_0,FUN_08_CAT_0,FUN_09_CAT_0,MED_CAT_0,MED_01_CAT_0,MED_02_CAT_0,MED_03_CAT_0,MED_04_CAT_0,MED_NS_CAT_01,_SRC_FILE
0,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,Total,Total,4.5,...,4.3,4.3,4.3,6.0,6.2,6.1,5.4,NaN,NaN,MÉDIA HAD_MUNICIPIOS_2023.xlsx
1,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,Urbana,Total,4.5,...,4.3,4.3,4.3,6.1,6.4,6.2,5.4,NaN,NaN,MÉDIA HAD_MUNICIPIOS_2023.xlsx
2,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,Rural,Total,4.3,...,4.3,4.3,4.3,4.3,4.3,4.3,NaN,NaN,NaN,MÉDIA HAD_MUNICIPIOS_2023.xlsx
3,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,Total,Estadual,NaN,...,4.3,4.3,4.3,6.0,6.2,6.1,5.4,NaN,NaN,MÉDIA HAD_MUNICIPIOS_2023.xlsx
4,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,Urbana,Estadual,NaN,...,4.3,4.3,4.3,6.1,6.4,6.2,5.4,NaN,NaN,MÉDIA HAD_MUNICIPIOS_2023.xlsx



## 5. HAD — Escolas (2023-2025)

Mesma lógica dos municípios (a base já traz `SG_UF` e `CO_MUNICIPIO`), mas
com o grão adicional de `CO_ENTIDADE` (código INEP da escola). Como são
~172 mil linhas por ano (~517 mil empilhadas), o formato `.parquet` aqui
ajuda bastante (bem mais compacto e rápido de ler do que `.csv`).


In [13]:

arquivos_had_esc = {
    2023: "MÉDIA HAD_ESCOLAS_2023.xlsx",
    2024: "MÉDIA HAD_ESCOLAS_2024.xlsx",
    2025: "MÉDIA HAD_ESCOLAS_2025.xlsx",
}

frames = []
for ano, arq in arquivos_had_esc.items():
    df = pd.read_excel(inp(arq), sheet_name=0, header=8)
    df["_SRC_FILE"] = arq
    frames.append(df)

bronze_had_esc = pd.concat(frames, ignore_index=True)
bronze_had_esc = bronze_had_esc[
    ~bronze_had_esc["NU_ANO_CENSO"].astype(str).str.contains("Fonte|Nota", na=False)
]
bronze_had_esc["NU_ANO_CENSO"] = pd.to_numeric(bronze_had_esc["NU_ANO_CENSO"], errors="coerce")
bronze_had_esc = bronze_had_esc.dropna(subset=["NU_ANO_CENSO"])

# colunas de indicador ficam como string no bronze pois misturam "--" e número
ind_cols_esc = [c for c in bronze_had_esc.columns if "_CAT_0" in c]
for c in ind_cols_esc:
    bronze_had_esc[c] = bronze_had_esc[c].astype(str)
salvar_parquet(bronze_had_esc, os.path.join(BRONZE_DIR, "had_escolas_2023_2025.parquet"))

# ---------------------------- SILVER ----------------------------------
silver_had_esc = bronze_had_esc.rename(columns={"NU_ANO_CENSO": "ANO"}).copy()
silver_had_esc = silver_had_esc.merge(dim_uf, on="SG_UF", how="left")

for c in ind_cols_esc:
    silver_had_esc[c] = pd.to_numeric(silver_had_esc[c].replace("--", pd.NA), errors="coerce")

silver_had_esc = silver_had_esc.rename(
    columns={"NO_REGIAO_x": "NO_REGIAO", "NO_CATEGORIA": "LOCALIZACAO", "NO_DEPENDENCIA": "DEPENDENCIA_ADM"}
)
cols_final = (
    ["ANO", "COD_REGIAO", "NO_REGIAO", "COD_UF", "SG_UF", "NO_UF", "CO_MUNICIPIO", "NO_MUNICIPIO",
     "CO_ENTIDADE", "NO_ENTIDADE", "LOCALIZACAO", "DEPENDENCIA_ADM"] + ind_cols_esc + ["_SRC_FILE"]
)
silver_had_esc = silver_had_esc[[c for c in cols_final if c in silver_had_esc.columns]]

salvar_parquet(silver_had_esc, os.path.join(SILVER_DIR, "had_escolas_2023_2025.parquet"))

print("HAD Escolas:", silver_had_esc.shape)
print("Linhas sem COD_UF:", silver_had_esc["COD_UF"].isna().sum())
silver_had_esc.head()


HAD Escolas: (517080, 33)
Linhas sem COD_UF: 0


,ANO,NO_REGIAO,COD_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,CO_ENTIDADE,NO_ENTIDADE,LOCALIZACAO,...,FUN_07_CAT_0,FUN_08_CAT_0,FUN_09_CAT_0,MED_CAT_0,MED_01_CAT_0,MED_02_CAT_0,MED_03_CAT_0,MED_04_CAT_0,MED_NS_CAT_0,_SRC_FILE
0,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,11022558.0,EIEEF HAP BITT TUPARI,Rural,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MÉDIA HAD_ESCOLAS_2023.xlsx
1,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,11024372.0,EMEIEF ANA NERY,Urbana,...,4.3,4.3,4.3,NaN,NaN,NaN,NaN,NaN,NaN,MÉDIA HAD_ESCOLAS_2023.xlsx
2,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,11024666.0,EMEIEF BOA ESPERANCA,Rural,...,4.3,4.3,4.3,NaN,NaN,NaN,NaN,NaN,NaN,MÉDIA HAD_ESCOLAS_2023.xlsx
3,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,11024682.0,EEEFM EURIDICE LOPES PEDROSO,Urbana,...,4.3,4.4,NaN,4.9,5.3,5.3,4.3,NaN,NaN,MÉDIA HAD_ESCOLAS_2023.xlsx
4,2023.0,Norte,11,RO,Rondônia,1100015.0,Alta Floresta D'Oeste,11024828.0,EMEIEF IZIDORO STEDILE,Urbana,...,4.3,4.3,4.3,NaN,NaN,NaN,NaN,NaN,NaN,MÉDIA HAD_ESCOLAS_2023.xlsx



## 6. IBGE/PNAD Contínua — Tabela 7109 (população residente, 6 a 9 anos)

Tabelas do IBGE (Sidra) vêm em formato "largo" (um ano por coluna) e com o
nome da unidade geográfica preenchido apenas na primeira linha de cada bloco
(merge de células no Excel original). O tratamento:

1. Recorta o cabeçalho (5 linhas de título/rótulos).
2. `ffill` na coluna de unidade geográfica para repetir o nome nas linhas de
   sub-categoria (grupo de idade).
3. `melt` das colunas de ano para formato longo (`ANO`, `VALOR`).
4. `adiciona_chaves_geo` para obter `COD_UF`/`SG_UF` (aqui as capitais vêm
   como `"Nome (SG)"`).


In [14]:

raw = pd.read_excel(inp("Tabela 7109 População residente 6a9 anos.xlsx"), sheet_name="Tabela", header=None)
salvar_parquet(raw, os.path.join(BRONZE_DIR, "ibge_tab7109_populacao_residente_raw.parquet"))

data = raw.iloc[5:].copy()
data.columns = ["UNIDADE_GEO", "GRUPO_IDADE", "2023", "2024", "2025"]
data["UNIDADE_GEO"] = data["UNIDADE_GEO"].ffill()
data = data[data["GRUPO_IDADE"].notna() & data["UNIDADE_GEO"].isin(GEO_VALIDOS)]

long_7109 = data.melt(
    id_vars=["UNIDADE_GEO", "GRUPO_IDADE"], value_vars=["2023", "2024", "2025"],
    var_name="ANO", value_name="POPULACAO_MIL",
)
long_7109["ANO"] = long_7109["ANO"].astype(int)
long_7109["POPULACAO_MIL"] = pd.to_numeric(long_7109["POPULACAO_MIL"], errors="coerce")
long_7109 = adiciona_chaves_geo(long_7109, col_geo="UNIDADE_GEO")
long_7109 = long_7109.rename(columns={"GRUPO_IDADE": "FAIXA_ETARIA"})
long_7109 = long_7109[["ANO", "TP_UNIDGEO", "UNIDADE_GEO", "COD_UF", "SG_UF", "FAIXA_ETARIA", "POPULACAO_MIL"]]

salvar_parquet(long_7109, os.path.join(SILVER_DIR, "ibge_tab7109_populacao_residente_6a9_long.parquet"))
print("Tabela 7109:", long_7109.shape)
print("UF/Capital sem COD_UF:",
      long_7109.loc[long_7109["TP_UNIDGEO"].isin(["UF", "CAPITAL"]), "COD_UF"].isna().sum())
long_7109.head()


Tabela 7109: (168, 7)
UF/Capital sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDADE_GEO,COD_UF,SG_UF,FAIXA_ETARIA,POPULACAO_MIL
0,2023,BRASIL,Brasil,NaN,<NA>,Total,210980
1,2023,BRASIL,Brasil,NaN,<NA>,6 a 9 anos,11632
2,2023,UF,Rondônia,11.0,RO,Total,1728
3,2023,UF,Rondônia,11.0,RO,6 a 9 anos,94
4,2023,UF,Acre,12.0,AC,Total,857



## 7. IBGE/PNAD Contínua — Tabela 7113 (taxa de analfabetismo, 15+ anos)

Mesma lógica da Tabela 7109 (esta tabela não desce ao nível de capitais,
apenas Brasil e UFs).


In [15]:

raw = pd.read_excel(inp("Tabela 7113  Taxa de analfabetismo.xlsx"), sheet_name="Tabela", header=None)
salvar_parquet(raw, os.path.join(BRONZE_DIR, "ibge_tab7113_taxa_analfabetismo_raw.parquet"))

data = raw.iloc[5:].copy()
data.columns = ["UNIDADE_GEO", "GRUPO_IDADE", "2023", "2024", "2025"]
data["UNIDADE_GEO"] = data["UNIDADE_GEO"].ffill()
data = data[data["GRUPO_IDADE"].notna() & data["UNIDADE_GEO"].isin(GEO_VALIDOS)]

long_7113 = data.melt(
    id_vars=["UNIDADE_GEO", "GRUPO_IDADE"], value_vars=["2023", "2024", "2025"],
    var_name="ANO", value_name="TAXA_ANALFABETISMO_PCT",
)
long_7113["ANO"] = long_7113["ANO"].astype(int)
long_7113["TAXA_ANALFABETISMO_PCT"] = pd.to_numeric(long_7113["TAXA_ANALFABETISMO_PCT"], errors="coerce")
long_7113 = adiciona_chaves_geo(long_7113, col_geo="UNIDADE_GEO")
long_7113 = long_7113.rename(columns={"GRUPO_IDADE": "FAIXA_ETARIA"})
long_7113 = long_7113[["ANO", "TP_UNIDGEO", "UNIDADE_GEO", "COD_UF", "SG_UF", "FAIXA_ETARIA", "TAXA_ANALFABETISMO_PCT"]]

salvar_parquet(long_7113, os.path.join(SILVER_DIR, "ibge_tab7113_taxa_analfabetismo_long.parquet"))
print("Tabela 7113:", long_7113.shape)
print("UF sem COD_UF:", long_7113.loc[long_7113["TP_UNIDGEO"] == "UF", "COD_UF"].isna().sum())
long_7113.head()


Tabela 7113: (336, 7)
UF sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDADE_GEO,COD_UF,SG_UF,FAIXA_ETARIA,TAXA_ANALFABETISMO_PCT
0,2023,BRASIL,Brasil,NaN,<NA>,15 anos ou mais,5.4
1,2023,BRASIL,Brasil,NaN,<NA>,18 anos ou mais,5.7
2,2023,BRASIL,Brasil,NaN,<NA>,25 anos ou mais,6.5
3,2023,BRASIL,Brasil,NaN,<NA>,40 anos ou mais,9.4
4,2023,UF,Rondônia,11.0,RO,15 anos ou mais,5.1



## 8. IBGE/PNAD Contínua — Tabela 4636 (rendimento médio mensal, R$)


In [16]:

raw = pd.read_excel(inp("Tabela 4636 Rendimento médio mensal uf.xlsx"), sheet_name="Tabela", header=None)
salvar_parquet(raw, os.path.join(BRONZE_DIR, "ibge_tab4636_rendimento_medio_raw.parquet"))

data = raw.iloc[4:].copy()
data.columns = ["UNIDADE_GEO", "2023", "2024", "2025"]
data = data[data["UNIDADE_GEO"].isin(GEO_VALIDOS)]

long_4636 = data.melt(
    id_vars=["UNIDADE_GEO"], value_vars=["2023", "2024", "2025"],
    var_name="ANO", value_name="RENDIMENTO_MEDIO_MENSAL_REAIS",
)
long_4636["ANO"] = long_4636["ANO"].astype(int)
long_4636["RENDIMENTO_MEDIO_MENSAL_REAIS"] = pd.to_numeric(long_4636["RENDIMENTO_MEDIO_MENSAL_REAIS"], errors="coerce")
long_4636 = adiciona_chaves_geo(long_4636, col_geo="UNIDADE_GEO")
long_4636 = long_4636[["ANO", "TP_UNIDGEO", "UNIDADE_GEO", "COD_UF", "SG_UF", "RENDIMENTO_MEDIO_MENSAL_REAIS"]]

salvar_parquet(long_4636, os.path.join(SILVER_DIR, "ibge_tab4636_rendimento_medio_mensal_long.parquet"))
print("Tabela 4636:", long_4636.shape)
print("UF sem COD_UF:", long_4636.loc[long_4636["TP_UNIDGEO"] == "UF", "COD_UF"].isna().sum())
long_4636.head()


Tabela 4636: (84, 6)
UF sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDADE_GEO,COD_UF,SG_UF,RENDIMENTO_MEDIO_MENSAL_REAIS
0,2023,BRASIL,Brasil,NaN,<NA>,3276
1,2023,UF,Rondônia,11.0,RO,2916
2,2023,UF,Acre,12.0,AC,2599
3,2023,UF,Amazonas,13.0,AM,2549
4,2023,UF,Roraima,14.0,RR,2992



## 9. PNAD Contínua — Tabela 2.2 (rendimento domiciliar per capita sem benefícios, por sexo/raça)

Arquivo `.xls` (formato binário antigo — exige `xlrd`) com **uma aba por ano**
(o arquivo original traz série histórica desde 2012, mas este pipeline usa só
**2023 em diante**, para ficar na mesma janela temporal do restante da base)
e cabeçalho hierárquico de 3 linhas (recorte → sub-recorte → medida
Médio/Mediano). O tratamento:

1. Itera por todas as abas cujo nome é um ano (ignora as abas `"(CV)"`,
   que trazem o coeficiente de variação, não o valor).
2. Reconstrói os nomes de coluna combinando as linhas de cabeçalho
   (`"<recorte>|<medida>"`).
3. Empilha todos os anos e depois `melt` para formato longo.
4. Aqui as capitais aparecem **sem sigla entre parênteses** (diferente das
   tabelas 7109/7113/4636), então `adiciona_chaves_geo` resolve pelo nome
   exato da capital (`dim_uf.NO_CAPITAL`).


In [17]:

xl = pd.ExcelFile(inp("Tabela 2.2 (RendCaract_Geo_sbenef).xls"))
# mantém só 2023 em diante — o arquivo original tem série histórica desde 2012,
# mas o recorte de análise deste pipeline começa em 2023 (mesma janela do HAD)
abas_ano = [s for s in xl.sheet_names if s.isdigit() and int(s) >= 2023]

frames = []
for aba in abas_ano:
    raw = xl.parse(aba, header=None)
    linha_categoria = raw.iloc[4].ffill()
    linha_medida = raw.iloc[5]
    nomes_colunas = []
    for i in range(1, raw.shape[1]):
        categoria = linha_categoria[i] if pd.notna(linha_categoria[i]) else "Total"
        medida = linha_medida[i]
        nomes_colunas.append(f"{categoria}|{medida}")

    data = raw.iloc[6:].copy()
    data.columns = ["UNIDADE_GEO"] + nomes_colunas
    data = data[data["UNIDADE_GEO"].isin(GEO_VALIDOS)]
    data["ANO"] = int(aba)
    frames.append(data)

bronze_tab22 = pd.concat(frames, ignore_index=True)
salvar_parquet(bronze_tab22, os.path.join(BRONZE_DIR, "pnad_tab2_2_rend_percapita_sbenef_raw.parquet"))

# ---------------------------- SILVER ----------------------------------
id_vars = ["ANO", "UNIDADE_GEO"]
value_vars = [c for c in bronze_tab22.columns if c not in id_vars]
long_tab22 = bronze_tab22.melt(id_vars=id_vars, value_vars=value_vars,
                                var_name="CATEGORIA_MEDIDA", value_name="VALOR")
long_tab22[["RECORTE_SEXO_RACA", "MEDIDA"]] = long_tab22["CATEGORIA_MEDIDA"].str.split("|", expand=True)
long_tab22 = long_tab22.drop(columns=["CATEGORIA_MEDIDA"])
long_tab22["VALOR"] = pd.to_numeric(long_tab22["VALOR"], errors="coerce")
long_tab22 = adiciona_chaves_geo(long_tab22, col_geo="UNIDADE_GEO")

long_tab22 = long_tab22[["ANO", "TP_UNIDGEO", "UNIDADE_GEO", "COD_UF", "SG_UF",
                          "RECORTE_SEXO_RACA", "MEDIDA", "VALOR"]]
salvar_parquet(long_tab22, os.path.join(SILVER_DIR, "pnad_tab2_2_rend_percapita_sbenef_long.parquet"))

print("Tabela 2.2:", long_tab22.shape, "| anos:", sorted(long_tab22["ANO"].unique()))
print("UF/Capital sem COD_UF:",
      long_tab22.loc[long_tab22["TP_UNIDGEO"].isin(["UF", "CAPITAL"]), "COD_UF"].isna().sum())
long_tab22.head()


Tabela 2.2: (2640, 8) | anos: [np.int64(2023), np.int64(2024)]
UF/Capital sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDADE_GEO,COD_UF,SG_UF,RECORTE_SEXO_RACA,MEDIDA,VALOR
0,2024,BRASIL,Brasil,NaN,<NA>,Total,Médio,1940.357446
1,2024,REGIAO,Norte,NaN,<NA>,Total,Médio,1274.869852
2,2024,UF,Rondônia,11.0,RO,Total,Médio,1652.756338
3,2024,CAPITAL,Porto Velho,11.0,RO,Total,Médio,1806.939243
4,2024,UF,Acre,12.0,AC,Total,Médio,1135.136958



## 10. PNAD Contínua — Tabela 4.3 (taxa de frequência escolar líquida ajustada)

Mesmo formato de arquivo da Tabela 2.2 (uma aba por ano), porém com cabeçalho
mais simples: 5 colunas fixas de faixa etária/nível de ensino (sem recorte de
sexo/raça). O arquivo original traz 2016 a 2019 e 2022 a 2024 (sem 2020/2021,
período de suspensão da pesquisa suplementar durante a pandemia, e sem 2015);
este pipeline mantém apenas **2023 em diante**, pelo mesmo motivo da Tabela 2.2.


In [18]:

xl43 = pd.ExcelFile(inp("Tabela 4.3 (FreqLiq_Geo).xls"))
# mantém só 2023 em diante, pelo mesmo motivo da Tabela 2.2
abas_ano_43 = [s for s in xl43.sheet_names if s.isdigit() and int(s) >= 2023]

colunas_43 = [
    "UNIDADE_GEO",
    "6 a 14 anos, no ensino fundamental",
    "6 a 10 anos, nos anos iniciais do ensino fundamental",
    "11 a 14 anos, nos anos finais do ensino fundamental",
    "15 a 17 anos, no ensino médio",
    "18 a 24 anos, no ensino superior",
]

frames = []
for aba in abas_ano_43:
    raw = xl43.parse(aba, header=None)
    data = raw.iloc[8:].copy()  # linhas de dado começam após o bloco de cabeçalho
    data = data.iloc[:, :len(colunas_43)]
    data.columns = colunas_43
    data = data[data["UNIDADE_GEO"].isin(GEO_VALIDOS)]
    data["ANO"] = int(aba)
    frames.append(data)

bronze_tab43 = pd.concat(frames, ignore_index=True)
salvar_parquet(bronze_tab43, os.path.join(BRONZE_DIR, "pnad_tab4_3_freq_liquida_raw.parquet"))

# ---------------------------- SILVER ----------------------------------
id_vars = ["ANO", "UNIDADE_GEO"]
value_vars = [c for c in colunas_43 if c != "UNIDADE_GEO"]
long_tab43 = bronze_tab43.melt(id_vars=id_vars, value_vars=value_vars,
                                var_name="NIVEL_FAIXA_ETARIA", value_name="TAXA_FREQ_LIQUIDA_PCT")
long_tab43["TAXA_FREQ_LIQUIDA_PCT"] = pd.to_numeric(long_tab43["TAXA_FREQ_LIQUIDA_PCT"], errors="coerce")
long_tab43 = adiciona_chaves_geo(long_tab43, col_geo="UNIDADE_GEO")
long_tab43 = long_tab43[["ANO", "TP_UNIDGEO", "UNIDADE_GEO", "COD_UF", "SG_UF",
                          "NIVEL_FAIXA_ETARIA", "TAXA_FREQ_LIQUIDA_PCT"]]

salvar_parquet(long_tab43, os.path.join(SILVER_DIR, "pnad_tab4_3_freq_liquida_long.parquet"))
print("Tabela 4.3:", long_tab43.shape, "| anos:", sorted(long_tab43["ANO"].unique()))
print("UF/Capital sem COD_UF:",
      long_tab43.loc[long_tab43["TP_UNIDGEO"].isin(["UF", "CAPITAL"]), "COD_UF"].isna().sum())
long_tab43.head()


Tabela 4.3: (600, 7) | anos: [np.int64(2023), np.int64(2024)]
UF/Capital sem COD_UF: 0


,ANO,TP_UNIDGEO,UNIDADE_GEO,COD_UF,SG_UF,NIVEL_FAIXA_ETARIA,TAXA_FREQ_LIQUIDA_PCT
0,2024,BRASIL,Brasil,NaN,<NA>,"6 a 14 anos, no ensino fundamental",94.566864
1,2024,REGIAO,Norte,NaN,<NA>,"6 a 14 anos, no ensino fundamental",94.288228
2,2024,UF,Rondônia,11.0,RO,"6 a 14 anos, no ensino fundamental",94.750387
3,2024,CAPITAL,Porto Velho,11.0,RO,"6 a 14 anos, no ensino fundamental",93.686075
4,2024,UF,Acre,12.0,AC,"6 a 14 anos, no ensino fundamental",93.525464



## 11. Microdados do Painel Estratégico de Priorização (CNCA)

Base já vem no grão de escola, com colunas de risco em UF, Regional,
Município e Escola (escala 1–5). O tratamento:

1. Pula as duas primeiras linhas (título + disclaimer) e usa a 3ª como
   cabeçalho real.
2. Remove a linha de rodapé (`Fonte: CGMA/Dimam/SEB/MEC...`).
3. **Decodifica cada uma das 4 colunas de risco** usando `dim_risco`
   (construída a partir do item 4.3.2.9 da Nota Técnica), criando colunas
   `_CATEGORIA` e `_DESC` para cada nível.
4. Os nulos em `NOME DA REGIONAL` / `RISCO DA REGIONAL` / `RISCO DO MUNICÍPIO`
   correspondem, conforme a própria Nota Técnica (item 4.3.2.7), à exceção
   metodológica do **Distrito Federal** (que não possui municípios e usa as
   regionais do Censo Escolar como unidade equivalente) — são preservados
   como nulos e não descartados.


In [19]:

raw_prior = pd.read_excel(
    inp("Microdados do Painel Estratégico de Priorização.xlsx"),
    sheet_name="BASE_FINAL_PRIORIZACAO", header=2,
)
salvar_parquet(raw_prior, os.path.join(BRONZE_DIR, "microdados_priorizacao_raw.parquet"))

# remove linha de rodapé (Fonte: ...)
silver_prior = raw_prior[~raw_prior["REGIÃO"].astype(str).str.contains("Fonte", na=False)].copy()

renomeia = {
    "REGIÃO": "NO_REGIAO",
    "UF": "NO_UF",
    "SIGLA DA UF": "SG_UF",
    "CÓDIGO DA UF": "COD_UF",
    "Risco da UF": "COD_RISCO_UF",
    "NOME DA REGIONAL": "NO_REGIONAL",
    "RISCO DA REGIONAL": "COD_RISCO_REGIONAL",
    "CÓDIGO DO MUNICÍPIO": "CO_MUNICIPIO",
    "NOME DO MUNICIPIO": "NO_MUNICIPIO",
    "RISCO DO MUNICÍPIO": "COD_RISCO_MUNICIPIO",
    "CÓDIGO DA ESCOLA": "CO_ENTIDADE",
    "NOME DA ESCOLA": "NO_ENTIDADE",
    "RISCO DA ESCOLA": "COD_RISCO_ESCOLA",
    "REDE": "DEPENDENCIA_ADM",
    "NOME DA REGIONAL DE ENSINO - CENSO ESCOLAR": "NO_REGIONAL_CENSO",
    "MATRÍCULA DO 2º ANO DO EF": "QT_MATRICULA_2ANO_EF",
}
silver_prior = silver_prior.rename(columns=renomeia)
silver_prior["COD_UF"] = pd.to_numeric(silver_prior["COD_UF"], errors="coerce")
silver_prior["CO_MUNICIPIO"] = pd.to_numeric(silver_prior["CO_MUNICIPIO"], errors="coerce")
silver_prior["CO_ENTIDADE"] = pd.to_numeric(silver_prior["CO_ENTIDADE"], errors="coerce")
silver_prior["QT_MATRICULA_2ANO_EF"] = pd.to_numeric(silver_prior["QT_MATRICULA_2ANO_EF"], errors="coerce")

# ---- decodifica as 4 colunas de risco usando dim_risco ----
mapa_categoria = dict(zip(dim_risco["COD_RISCO"], dim_risco["CATEGORIA_RISCO"]))
mapa_nivel_desc = dict(zip(dim_risco["COD_RISCO"], dim_risco["NIVEL_RISCO_DESC"]))
mapa_impacto = dict(zip(dim_risco["COD_RISCO"], dim_risco["DESC_IMPACTO"]))

for nivel in ["UF", "REGIONAL", "MUNICIPIO", "ESCOLA"]:
    col_cod = f"COD_RISCO_{nivel}"
    silver_prior[f"CATEGORIA_RISCO_{nivel}"] = silver_prior[col_cod].map(mapa_categoria)
    silver_prior[f"NIVEL_RISCO_DESC_{nivel}"] = silver_prior[col_cod].map(mapa_nivel_desc)
    silver_prior[f"DESC_IMPACTO_{nivel}"] = silver_prior[col_cod].map(mapa_impacto)

salvar_parquet(silver_prior, os.path.join(SILVER_DIR, "microdados_priorizacao_clean.parquet"))

print("Microdados de Priorização:", silver_prior.shape)
print("Nulos em RISCO/REGIONAL (esperado = casos do DF):",
      silver_prior[["NO_REGIONAL", "COD_RISCO_REGIONAL", "COD_RISCO_MUNICIPIO"]].isna().sum().to_dict())
silver_prior.head()


Microdados de Priorização: (43521, 28)
Nulos em RISCO/REGIONAL (esperado = casos do DF): {'NO_REGIONAL': 264, 'COD_RISCO_REGIONAL': 264, 'COD_RISCO_MUNICIPIO': 264}


,NO_REGIAO,NO_UF,SG_UF,COD_UF,COD_RISCO_UF,NO_REGIONAL,COD_RISCO_REGIONAL,CO_MUNICIPIO,NO_MUNICIPIO,COD_RISCO_MUNICIPIO,...,DESC_IMPACTO_UF,CATEGORIA_RISCO_REGIONAL,NIVEL_RISCO_DESC_REGIONAL,DESC_IMPACTO_REGIONAL,CATEGORIA_RISCO_MUNICIPIO,NIVEL_RISCO_DESC_MUNICIPIO,DESC_IMPACTO_MUNICIPIO,CATEGORIA_RISCO_ESCOLA,NIVEL_RISCO_DESC_ESCOLA,DESC_IMPACTO_ESCOLA
0,Norte,Rondônia,RO,11.0,5.0,Superintendência Regional de Educação de Porto...,1.0,1100205.0,Porto Velho,1.0,...,Grande número de estudantes impactados (mais d...,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Moderado,3 – Risco moderado de não alcançar a meta,Impacto em um grupo médio de estudantes
1,Norte,Rondônia,RO,11.0,5.0,Superintendência Regional de Educação de Porto...,1.0,1100205.0,Porto Velho,1.0,...,Grande número de estudantes impactados (mais d...,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Moderado,3 – Risco moderado de não alcançar a meta,Impacto em um grupo médio de estudantes
2,Norte,Rondônia,RO,11.0,5.0,Superintendência Regional de Educação de Porto...,1.0,1100205.0,Porto Velho,1.0,...,Grande número de estudantes impactados (mais d...,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Baixo,2 – Risco moderado-baixo de não alcançar a meta,Impacto em um grupo reduzido de estudantes
3,Norte,Rondônia,RO,11.0,5.0,Superintendência Regional de Educação de Porto...,1.0,1100205.0,Porto Velho,1.0,...,Grande número de estudantes impactados (mais d...,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Alto,4 – Risco moderado-alto de não alcançar a meta,Muitos estudantes impactados
4,Norte,Rondônia,RO,11.0,5.0,Superintendência Regional de Educação de Porto...,1.0,1100205.0,Porto Velho,1.0,...,Grande número de estudantes impactados (mais d...,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Muito baixo,1 – Baixo risco de não alcançar a meta,Poucos estudantes impactados,Moderado,3 – Risco moderado de não alcançar a meta,Impacto em um grupo médio de estudantes



## 12. Camada Gold — tabelas de fato prontas para cruzamento / modelagem

Monta três tabelas de fato, uma por grão, já cruzando as fontes silver
entre si pela chave `COD_UF` (e `CO_MUNICIPIO` / `CO_ENTIDADE` quando
aplicável):

- **`fact_uf_ano`**: HAD (UF, filtro `Total`/`Total`) + população 6-9 anos +
  taxa de analfabetismo + rendimento médio mensal + rendimento per capita
  (sexo/raça = Total) + frequência líquida — tudo no grão UF × Ano.
- **`fact_municipio`**: HAD município (filtro `Total`/`Total`) + risco do
  Município/UF vindos do Painel de Priorização, no grão Município. As
  variáveis do IBGE/PNAD (população, analfabetismo, rendimento etc.) **não
  existem nessa granularidade** — a PNAD Contínua não tem amostra suficiente
  para estimar esses indicadores por município. Por isso, nesse caso o
  cruzamento é feito **por UF** (mesmo `COD_UF` + `ANO`) e as colunas entram
  com o prefixo `UF_`, deixando explícito que é um valor agregado do estado,
  repetido para todos os municípios daquela UF — e não uma medida local.
- **`fact_escola`**: mesma lógica de `fact_municipio`, no grão Escola —
  HAD escola + risco da Escola/Município/UF do Painel de Priorização +
  contexto socioeconômico da UF (`UF_...`) cruzado por `COD_UF` + `ANO`.

> Ajuste os filtros de `LOCALIZACAO`/`DEPENDENCIA_ADM` conforme a necessidade
> do seu modelo — aqui usamos o total consolidado como exemplo de junção.


In [20]:

# ---------------- Contexto socioeconômico de UF (fonte única, reaproveitada) ----------------
# Uma única tabela ANO x COD_UF com todos os indicadores do IBGE/PNAD, para
# evitar repetir os pivots/filtros em cada tabela de fato que precisar dela.
analf_uf = long_7113[(long_7113["TP_UNIDGEO"] == "UF") & (long_7113["FAIXA_ETARIA"] == "15 anos ou mais")]
analf_uf = analf_uf.pivot_table(index=["ANO", "COD_UF"], values="TAXA_ANALFABETISMO_PCT", aggfunc="first").reset_index()

pop_uf = long_7109[(long_7109["TP_UNIDGEO"] == "UF") & (long_7109["FAIXA_ETARIA"] == "6 a 9 anos")]
pop_uf = pop_uf.pivot_table(index=["ANO", "COD_UF"], values="POPULACAO_MIL", aggfunc="first").reset_index()

rend_uf = long_4636[long_4636["TP_UNIDGEO"] == "UF"][["ANO", "COD_UF", "RENDIMENTO_MEDIO_MENSAL_REAIS"]]

rend_percapita_uf = long_tab22[
    (long_tab22["TP_UNIDGEO"] == "UF") &
    (long_tab22["RECORTE_SEXO_RACA"] == "Total") &
    (long_tab22["MEDIDA"] == "Médio")
][["ANO", "COD_UF", "VALOR"]].rename(columns={"VALOR": "RENDA_PERCAPITA_SBENEF_MEDIO"})

freq_uf = long_tab43[
    (long_tab43["TP_UNIDGEO"] == "UF") &
    (long_tab43["NIVEL_FAIXA_ETARIA"] == "6 a 10 anos, no ensino fundamental")
][["ANO", "COD_UF", "TAXA_FREQ_LIQUIDA_PCT"]].rename(columns={"TAXA_FREQ_LIQUIDA_PCT": "TAXA_FREQ_LIQUIDA_FUND_PCT"})

contexto_uf = pop_uf.merge(analf_uf, on=["ANO", "COD_UF"], how="outer")
contexto_uf = contexto_uf.merge(rend_uf, on=["ANO", "COD_UF"], how="outer")
contexto_uf = contexto_uf.merge(rend_percapita_uf, on=["ANO", "COD_UF"], how="outer")
contexto_uf = contexto_uf.merge(freq_uf, on=["ANO", "COD_UF"], how="outer")

salvar_parquet(contexto_uf, os.path.join(GOLD_DIR, "dim_contexto_socioeconomico_uf.parquet"))
print("contexto_uf (ANO x COD_UF):", contexto_uf.shape)

# versão com prefixo UF_, para deixar claro nas tabelas de município/escola
# que esses valores são um agregado estadual repetido, não uma medida local
contexto_uf_prefixado = contexto_uf.rename(columns={
    c: f"UF_{c}" for c in contexto_uf.columns if c not in ("ANO", "COD_UF")
})

# ---------------- fact_uf_ano ----------------
had_uf_total = silver_had_uf[
    (silver_had_uf["TP_UNIDGEO"] == "UF") &
    (silver_had_uf["LOCALIZACAO"] == "Total") &
    (silver_had_uf["DEPENDENCIA_ADM"] == "Total")
].copy()

fact_uf_ano = had_uf_total[["ANO", "COD_REGIAO", "NO_REGIAO", "COD_UF", "SG_UF", "NO_UF" if "NO_UF" in had_uf_total.columns else "UNIDGEO"] + ind_cols]
fact_uf_ano = fact_uf_ano.merge(contexto_uf, on=["ANO", "COD_UF"], how="left")

salvar_parquet(fact_uf_ano, os.path.join(GOLD_DIR, "fact_uf_ano.parquet"))
print("fact_uf_ano:", fact_uf_ano.shape)

# ---------------- fact_municipio ----------------
had_mun_total = silver_had_mun[
    (silver_had_mun["LOCALIZACAO"] == "Total") & (silver_had_mun["DEPENDENCIA_ADM"] == "Total")
].copy()

risco_mun = silver_prior[
    ["CO_MUNICIPIO", "COD_UF", "COD_RISCO_UF", "CATEGORIA_RISCO_UF",
     "COD_RISCO_MUNICIPIO", "CATEGORIA_RISCO_MUNICIPIO", "NIVEL_RISCO_DESC_MUNICIPIO"]
].drop_duplicates(subset=["CO_MUNICIPIO"])

fact_municipio = had_mun_total.merge(risco_mun, on=["CO_MUNICIPIO", "COD_UF"], how="left")
# sem chave de município nas fontes do IBGE/PNAD -> cruzamento por UF (+ANO)
fact_municipio = fact_municipio.merge(contexto_uf_prefixado, on=["ANO", "COD_UF"], how="left")

salvar_parquet(fact_municipio, os.path.join(GOLD_DIR, "fact_municipio.parquet"))
print("fact_municipio:", fact_municipio.shape)

# ---------------- fact_escola ----------------
# No grão de escola, cada CO_ENTIDADE/ANO já é uma única linha (a coluna
# LOCALIZACAO indica se a própria escola é Rural/Urbana, não é um filtro de
# agregação como em UF/Município) — por isso aqui não filtramos por "Total".
had_esc_total = silver_had_esc.copy()

risco_esc = silver_prior[
    ["CO_ENTIDADE", "CO_MUNICIPIO", "COD_UF", "COD_RISCO_ESCOLA", "CATEGORIA_RISCO_ESCOLA",
     "NIVEL_RISCO_DESC_ESCOLA", "DESC_IMPACTO_ESCOLA", "QT_MATRICULA_2ANO_EF", "NO_REGIONAL_CENSO"]
].drop_duplicates(subset=["CO_ENTIDADE"])

fact_escola = had_esc_total.merge(risco_esc, on=["CO_ENTIDADE", "CO_MUNICIPIO", "COD_UF"], how="left")
# sem chave de município/escola nas fontes do IBGE/PNAD -> cruzamento por UF (+ANO)
fact_escola = fact_escola.merge(contexto_uf_prefixado, on=["ANO", "COD_UF"], how="left")

salvar_parquet(fact_escola, os.path.join(GOLD_DIR, "fact_escola.parquet"))
print("fact_escola:", fact_escola.shape)
print("Escolas com risco encontrado:", fact_escola["COD_RISCO_ESCOLA"].notna().sum(), "/", len(fact_escola))


contexto_uf (ANO x COD_UF): (85, 7)
fact_uf_ano: (85, 32)
fact_municipio: (18185, 41)
fact_escola: (597356, 44)
Escolas com risco encontrado: 147081 / 597356



## 13. Resumo dos arquivos gerados


In [27]:

def lista_arquivos(pasta):
    for raiz, _, arquivos in os.walk(pasta):
        for f in sorted(arquivos):
            caminho = os.path.join(raiz, f)
            tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
            print(f"  {caminho}  ({tamanho_mb:.2f} MB)")

print("BRONZE:")
lista_arquivos(BRONZE_DIR)
print("\nSILVER:")
lista_arquivos(SILVER_DIR)
print("\nGOLD:")
lista_arquivos(GOLD_DIR)


BRONZE:
  c:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_03\tech_challenge_03\src\data\bronze\had_brasil_regioes_ufs_2023_2025.parquet  (0.05 MB)
  c:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_03\tech_challenge_03\src\data\bronze\had_escolas_2023_2025.parquet  (16.53 MB)
  c:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_03\tech_challenge_03\src\data\bronze\had_municipios_2023_2025.parquet  (3.02 MB)
  c:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_03\tech_challenge_03\src\data\bronze\ibge_tab4636_rendimento_medio_raw.parquet  (0.00 MB)
  c:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_03\tech_challenge_03\src\data\bronze\ibge_tab7109_populacao_residente_raw.parquet  (0.01 MB)
  c:\Users\deth_\Carmel Capita

### Subir para o S3

In [ ]:
from src.data.utils import *
import boto3

session = iniciar_cessao_aws()
s3 = session.client("s3")

BUCKET = os.getenv("BUCKET_NAME")

pasta_atual = Path().resolve()
caminho_env = pasta_atual.parent / '.env'

dotenv.load_dotenv(caminho_env)


In [33]:
from pathlib import Path

# Aponta diretamente para a sua pasta src/data/gold
PASTA_GOLD = Path.cwd().parent / "src" / "data" / "gold"


for arquivo in PASTA_GOLD.glob("*.parquet"):
    chave_s3 = f"gold/dados_externos/{arquivo.name}"
    
    print(f"Enviando {arquivo.name} para s3://{BUCKET}/{chave_s3}...")
    
    s3.upload_file(
        Filename=str(arquivo),
        Bucket=BUCKET,
        Key=chave_s3
    )

print("\n✅ Todos os arquivos Gold foram enviados com sucesso!")


Enviando dim_contexto_socioeconomico_uf.parquet para s3://postech-challenge-datascience-003/gold/dados_externos/dim_contexto_socioeconomico_uf.parquet...
Enviando fact_escola.parquet para s3://postech-challenge-datascience-003/gold/dados_externos/fact_escola.parquet...
Enviando fact_municipio.parquet para s3://postech-challenge-datascience-003/gold/dados_externos/fact_municipio.parquet...
Enviando fact_uf_ano.parquet para s3://postech-challenge-datascience-003/gold/dados_externos/fact_uf_ano.parquet...

✅ Todos os arquivos Gold foram enviados com sucesso!
